# 03 - Scratch TextCNN

Input is `prompt [SEP] option`. Convolutional filters of sizes 3, 4, 5 with 64 channels each, global max-pool, dropout, sigmoid head. Class imbalance handled with `pos_weight=2.0`.

## Setup

In [1]:
import subprocess, sys
for pkg in ['wandb', 'pyarrow']:
    try: __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [2]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    DATA_DIR = ROOT / 'data'
    OUTPUT_DIR = ROOT

PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
FEATURES_DIR = OUTPUT_DIR / 'features'
FIGURES_DIR = OUTPUT_DIR / 'figures'
for p in [PREDICTIONS_DIR, FEATURES_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [3]:
import wandb
WANDB_ENTITY = '23f2001849-dl-genai-project'
WANDB_PROJECT = '23f2001849-t22026'
USE_WANDB = False
try:
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('WANDB_API_KEY')
    else:
        key = os.environ.get('WANDB_API_KEY', '').strip()
    if key:
        wandb.login(key=key, relogin=True)
        USE_WANDB = True
except Exception as e:
    print(f'W&B setup failed: {e}')
print(f'W&B enabled: {USE_WANDB}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Krishna\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B enabled: True


In [4]:
import json, time, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OPTIONS = ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES = 5
print(f'device: {device}')

device: cpu


## Config

In [5]:
MAX_LEN = 120
EMB_DIM = 128
N_FILTERS = 64
FILTER_SIZES = [3, 4, 5]
DROPOUT = 0.5
EPOCHS = 5
BATCH = 64
LR = 5e-4
WD = 1e-4
POS_WEIGHT = 2.0
VOCAB_MIN_FREQ = 2
FOLDS = 5

## Load features

In [6]:
train_long = pd.read_parquet(FEATURES_DIR / 'train_long.parquet')
test_long  = pd.read_parquet(FEATURES_DIR / 'test_long.parquet')
train_meta = pd.read_parquet(FEATURES_DIR / 'train_meta.parquet')
print(f'train_long: {train_long.shape}')

train_long: (10000, 24)


## Tokenize prompt + option

In [7]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

In [8]:
tr_texts = [f'{p} [SEP] {o}' for p, o in zip(train_long['prompt'].astype(str),
                                                train_long['option'].astype(str))]
te_texts = [f'{p} [SEP] {o}' for p, o in zip(test_long['prompt'].astype(str),
                                                test_long['option'].astype(str))]

In [9]:
counter = Counter()
for t in tr_texts + te_texts:
    counter.update(tokenize(t))

PAD_ID = 0
UNK_ID = 1
vocab = {'<pad>': PAD_ID, '<unk>': UNK_ID}
for word, freq in counter.most_common():
    if freq >= VOCAB_MIN_FREQ:
        vocab[word] = len(vocab)
print(f'vocab size: {len(vocab)}')

vocab size: 2976


In [10]:
def encode(text, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK_ID) for t in tokenize(text)[:max_len]]
    while len(ids) < max_len:
        ids.append(PAD_ID)
    return ids

train_ids = np.array([encode(t) for t in tr_texts], dtype=np.int64)
test_ids  = np.array([encode(t) for t in te_texts], dtype=np.int64)
print(f'train_ids: {train_ids.shape}, test_ids: {test_ids.shape}')

train_ids: (10000, 120), test_ids: (2500, 120)


## Model

In [11]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([
            nn.Conv1d(EMB_DIM, N_FILTERS, kernel_size=k, padding=k // 2)
            for k in FILTER_SIZES
        ])
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(N_FILTERS * len(FILTER_SIZES), 1)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        pooled = [F.relu(conv(emb)).max(dim=2).values for conv in self.convs]
        h = torch.cat(pooled, dim=1)
        return self.fc(self.dropout(h)).squeeze(-1)

In [12]:
class TokDataset(Dataset):
    def __init__(self, ids, y=None):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        if self.y is not None:
            return self.ids[i], self.y[i]
        return self.ids[i]

## Metric helpers

In [13]:
def map3_score(pred, true):
    s = 0.0
    for p, a in zip(pred, true):
        for k, L in enumerate(p[:3]):
            if L == a:
                s += 1.0 / (k + 1); break
    return s / len(true)

def scores_to_top3(grid):
    order = np.argsort(-grid, axis=1)
    return [[OPTIONS[i] for i in row[:3]] for row in order]

def long_to_grid(long_df, scores, ids_order):
    id_to_row = {q: i for i, q in enumerate(ids_order)}
    grid = np.zeros((len(ids_order), NUM_CHOICES))
    for i in range(len(long_df)):
        r = long_df.iloc[i]
        grid[id_to_row[r['id']], OPTIONS.index(r['letter'])] = scores[i]
    return grid

## Training helpers

In [14]:
def make_model(epochs, n_batches):
    torch.manual_seed(SEED)
    model = TextCNN(len(vocab)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * n_batches)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=device))
    return model, opt, sched, loss_fn

In [15]:
def train_epoch(model, dl, opt, sched, loss_fn):
    model.train()
    total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        sched.step()
        total += loss.item() * xb.size(0)
    return total / len(dl.dataset)

In [16]:
def eval_logits(model, dl):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(xb.to(device)).cpu() for xb, _ in dl])

### Question-level val metrics

Reshape per-option probabilities into a per-question `(N, 5)` grid so we can score `top1` and `MAP@3` alongside the binary loss.

In [17]:
def val_qgrid(fold_idx, val_probs):
    val_qs = train_meta.loc[train_meta['id'].isin(
        train_long.loc[train_long['fold'] == fold_idx, 'id'].unique()), 'id'].tolist()
    id_to_row = {q: i for i, q in enumerate(val_qs)}
    vdf = train_long[train_long['fold'] == fold_idx].reset_index(drop=True)
    grid = np.zeros((len(val_qs), NUM_CHOICES))
    for i in range(len(vdf)):
        r = vdf.iloc[i]
        grid[id_to_row[r['id']], OPTIONS.index(r['letter'])] = val_probs[i]
    letters = train_meta.set_index('id').loc[val_qs, 'answer'].tolist()
    return grid, letters

### Fold trainer

Best checkpoint per fold selected by val MAP@3.

In [18]:
def train_fold(X_tr, y_tr, X_val, y_val, fold_idx):
    tr_dl = DataLoader(TokDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
    va_dl = DataLoader(TokDataset(X_val, y_val), batch_size=256, shuffle=False)
    model, opt, sched, loss_fn = make_model(EPOCHS, len(tr_dl))
    val_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

    best_map3, best_probs = -1.0, None
    for ep in range(EPOCHS):
        tl = train_epoch(model, tr_dl, opt, sched, loss_fn)
        logits = eval_logits(model, va_dl)
        vl = float(val_loss_fn(logits, torch.tensor(y_val)))
        vprobs = torch.sigmoid(logits).numpy()
        grid, letters = val_qgrid(fold_idx, vprobs)
        v_top1 = float(np.mean([OPTIONS[grid[i].argmax()] == letters[i] for i in range(len(letters))]))
        v_map3 = map3_score(scores_to_top3(grid), letters)
        print(f'  fold {fold_idx} ep {ep+1}: train {tl:.4f}  val {vl:.4f}  top1 {v_top1:.4f}  map3 {v_map3:.4f}')
        if USE_WANDB:
            wandb.log({
                'epoch': ep,
                f'train_loss/fold_{fold_idx}': tl,
                f'val_loss/fold_{fold_idx}':   vl,
                f'val_top1/fold_{fold_idx}':   v_top1,
                f'val_map3/fold_{fold_idx}':   v_map3,
            })
        if v_map3 > best_map3:
            best_map3, best_probs = v_map3, vprobs.copy()
    return best_probs

## Run 5 folds

In [19]:
if USE_WANDB:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name='cnn', reinit=True,
               tags=['model:cnn', 'features:tokens'],
               config={'model': 'text_cnn', 'emb_dim': EMB_DIM, 'n_filters': N_FILTERS,
                       'filter_sizes': FILTER_SIZES, 'dropout': DROPOUT, 'epochs': EPOCHS,
                       'batch_size': BATCH, 'lr': LR, 'weight_decay': WD,
                       'pos_weight': POS_WEIGHT, 'vocab_size': len(vocab)})
    wandb.define_metric('epoch')
    for m in ['train_loss', 'val_loss', 'val_top1', 'val_map3']:
        wandb.define_metric(f'{m}/*', step_metric='epoch')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [20]:
y = train_long['is_correct'].astype(np.float32).values
folds = train_long['fold'].values
oof_cnn = np.zeros(len(train_long))

t0 = time.time()
for f in range(FOLDS):
    tr_m = folds != f
    va_m = folds == f
    oof_cnn[va_m] = train_fold(train_ids[tr_m], y[tr_m],
                                train_ids[va_m], y[va_m], f)

elapsed = time.time() - t0
print(f'training time: {elapsed:.1f}s')

  fold 0 ep 1: train 0.7530  val 0.7706  top1 0.2900  map3 0.4933
  fold 0 ep 2: train 0.6100  val 0.7513  top1 0.3650  map3 0.5483
  fold 0 ep 3: train 0.5193  val 0.7477  top1 0.3450  map3 0.5092
  fold 0 ep 4: train 0.4639  val 0.7553  top1 0.3600  map3 0.5279
  fold 0 ep 5: train 0.4544  val 0.7587  top1 0.3175  map3 0.5075
  fold 1 ep 1: train 0.7606  val 0.7670  top1 0.2550  map3 0.4204
  fold 1 ep 2: train 0.6273  val 0.7393  top1 0.3000  map3 0.4608
  fold 1 ep 3: train 0.5402  val 0.7512  top1 0.3275  map3 0.4738
  fold 1 ep 4: train 0.4817  val 0.7496  top1 0.3350  map3 0.4779
  fold 1 ep 5: train 0.4662  val 0.7504  top1 0.3350  map3 0.4808
  fold 2 ep 1: train 0.7554  val 0.7911  top1 0.3900  map3 0.4913
  fold 2 ep 2: train 0.6166  val 0.7606  top1 0.4100  map3 0.5363
  fold 2 ep 3: train 0.5253  val 0.7401  top1 0.3925  map3 0.5292
  fold 2 ep 4: train 0.4701  val 0.7489  top1 0.3900  map3 0.5267
  fold 2 ep 5: train 0.4520  val 0.7501  top1 0.3925  map3 0.5308
  fold 3 e

### Reading the metrics

**MAP@3** and **top1_acc** are the honest headline metrics for a 5-way MCQ.

**val_acc, precision, recall, F1** come from a binary `>0.5` threshold on per-option probabilities. The class distribution is 1 correct : 4 wrong per question, so the majority-class baseline is 80% accuracy. For softmax-normalized transformer probabilities the per-option mean is `1/5 = 0.2`, so almost nothing crosses 0.5 and `recall` collapses to near zero while `precision` becomes noisy. These numbers are reported for completeness — they are not diagnostic of model quality on this task.

## OOF metrics

In [21]:
true_letters = train_meta['answer'].tolist()
true_ids = train_meta['id'].tolist()
grid = long_to_grid(train_long, oof_cnn, true_ids)
top3 = scores_to_top3(grid)

oof_pred = (oof_cnn > 0.5).astype(int)
final = {
    'accuracy':  accuracy_score(y.astype(int), oof_pred),
    'precision': precision_score(y.astype(int), oof_pred, zero_division=0),
    'recall':    recall_score(y.astype(int), oof_pred, zero_division=0),
    'f1':        f1_score(y.astype(int), oof_pred, zero_division=0),
    'log_loss':  log_loss(y.astype(int), oof_cnn.clip(1e-7, 1 - 1e-7)),
    'top1_acc':  float(np.mean([t[0] == a for t, a in zip(top3, true_letters)])),
    'map3':      map3_score(top3, true_letters),
    'training_time_sec': elapsed,
}
print('CNN OOF metrics:')
for k, v in final.items():
    print(f'  {k}: {v:.4f}')

if USE_WANDB:
    for k, v in final.items():
        wandb.run.summary[k] = v
    wandb.finish()

CNN OOF metrics:
  accuracy: 0.8039
  precision: 0.9333
  recall: 0.0210
  f1: 0.0411
  log_loss: 0.5138
  top1_acc: 0.3495
  map3: 0.5041
  training_time_sec: 188.1276


epoch,▁▃▅▆█▁▃▅▆█▁▃▅▆█▁▃▅▆█▁▃▅▆█
train_loss/fold_0,█▅▃▁▁
train_loss/fold_1,█▅▃▁▁
train_loss/fold_2,█▅▃▁▁
train_loss/fold_3,█▅▃▁▁
train_loss/fold_4,█▅▃▂▁
val_loss/fold_0,█▂▁▃▄
val_loss/fold_1,█▁▄▄▄
val_loss/fold_2,█▄▁▂▂
val_loss/fold_3,▁▃▄██
+11,...


## Full-train and test predictions

In [22]:
full_dl = DataLoader(TokDataset(train_ids, y), batch_size=BATCH, shuffle=True)
model_full, opt, sched, loss_fn = make_model(EPOCHS, len(full_dl))
for ep in range(EPOCHS):
    tl = train_epoch(model_full, full_dl, opt, sched, loss_fn)
    print(f'  full ep {ep+1}: train {tl:.4f}')

  full ep 1: train 0.7523
  full ep 2: train 0.6122
  full ep 3: train 0.5197
  full ep 4: train 0.4536
  full ep 5: train 0.4339


In [23]:
model_full.eval()
test_dl = DataLoader(TokDataset(test_ids), batch_size=256, shuffle=False)
with torch.no_grad():
    logits = torch.cat([model_full(xb.to(device)).cpu() for xb in test_dl])
test_pred = torch.sigmoid(logits).numpy()
print(f'test predictions: {test_pred.shape}')

test predictions: (2500,)


## Save predictions

In [24]:
np.save(PREDICTIONS_DIR / 'oof_cnn.npy', oof_cnn)
np.save(PREDICTIONS_DIR / 'test_cnn.npy', test_pred)
with open(PREDICTIONS_DIR / 'cnn_metrics.json', 'w') as fh:
    json.dump(final, fh, indent=2)
print(f'saved oof_cnn.npy: {oof_cnn.shape}')
print(f'saved test_cnn.npy: {test_pred.shape}')

saved oof_cnn.npy: (10000,)
saved test_cnn.npy: (2500,)
